# T-REX 한식 인식 모델 학습 (YOLOv8n → TFLite)

AI Hub **74번 "음식 이미지 및 영양정보 텍스트"**(박스 어노테이션 포함, 400여 종)로 YOLOv8n을 파인튜닝하고, 앱(`app/src/main/assets/models/`)에 넣을 수 있는 형태로 변환하는 노트북이다.

### 실행 전 준비 (1회)
1. [AI Hub](https://aihub.or.kr)에서 74번 데이터셋 활용신청 → 승인 후 **마이페이지에서 API 키 발급**. (브라우저용 Innorix 다운로더는 쓰지 않는다 — Colab 안에서 `aihubshell`로 직접 받는다.)
2. Colab 메뉴 → 런타임 → 런타임 유형 변경 → **GPU** 선택 (Pro+ 권장: 디스크 200GB+, 긴 세션).
3. 아래 셀을 위에서부터 순서대로 실행.

### 데이터 크기와 디스크 전략
전체는 1TB 가까이지만 필요한 부분만 쓴다.

| 파일 | 크기 | 용도 |
|---|---|---|
| `[라벨]음식분류_TRAIN.zip` | 446MB | 클래스 목록·박스 좌표(JSON). **먼저 받아 구조를 확인한다.** |
| `[원천]음식분류_TRAIN_001~003.zip` | 27/32/28GB | 사진. **한 덩어리씩** 받아 목표 클래스만 640px로 줄여 저장하고 원본을 지운다. |
| `음식분류 AI 데이터 영양DB.xlsx` | 작음 | 앱 `foodDatabase` 의 근사값을 실제 값으로 교체 |
| `[원천]양추정_*` | 43GB+ | 인분/양 추정용 — **범위 밖, 받지 않는다** |

디스크 피크는 약 60GB(덩어리 1개 + 압축 해제), 최종 데이터셋은 3~5GB. 완성본은 Drive에 저장해 두면 재다운로드가 필요 없다.

### 산출물 (마지막 셀에서 zip으로 다운로드)
- `yolov8n_food.tflite` — INT8 양자화, **입출력은 float32 유지**. 입력 배치는 NHWC `[1,640,640,3]`가 기본이지만 앱 FoodDetector는 NCHW `[1,3,640,640]`도 처리한다(2026-09-09 변환본이 NCHW였다).
- `food_labels.txt` — 모델 클래스 인덱스 순서와 동일한 라벨 목록

두 파일을 `TREX_UI/app/src/main/assets/models/`에 덮어쓰고 빌드하면 앱이 자동으로 실추론으로 전환된다.

In [ ]:
# 1. 환경 설치
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# 2. AI Hub 다운로더(aihubshell) 설치 + 파일 목록 조회
# 출력에서 [라벨]음식분류_TRAIN.zip, [원천]음식분류_TRAIN_001~003.zip, 영양DB.xlsx 의 filekey 를 확인한다.
AIHUB_API_KEY = ''   # AI Hub 마이페이지에서 발급받은 API 키
DATASET_KEY = 74     # 음식 이미지 및 영양정보 텍스트

assert AIHUB_API_KEY, 'AIHUB_API_KEY 를 채운다'
!curl -s -o aihubshell https://api.aihub.or.kr/api/aihubshell.do && chmod +x aihubshell && mv aihubshell /usr/local/bin/
!aihubshell -mode l -datasetkey {DATASET_KEY} -aihubapikey "{AIHUB_API_KEY}"

In [ ]:
# 2b. 작은 파일부터: 라벨 zip(446MB) + 영양 DB(xlsx) 다운로드 → 구조 확인
# 2번 출력에서 읽은 filekey 를 채운다 (쉼표로 여러 개). 원천(사진) zip 은 아직 받지 않는다.
LABEL_FILEKEYS = ''   # 예: '12345,12346'  → [라벨]음식분류_TRAIN.zip, 음식분류 AI 데이터 영양DB.xlsx

assert LABEL_FILEKEYS, 'LABEL_FILEKEYS 를 채운다'
!mkdir -p /content/aihub_raw && cd /content/aihub_raw && aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {LABEL_FILEKEYS} -aihubapikey "{AIHUB_API_KEY}"

# 받은 파일·압축 내용·JSON 한 건을 출력한다. 이 출력을 보고 4번 변환 셀의 필드명(클래스·박스 좌표·이미지 파일명)을 확정한다.
import json, zipfile
from pathlib import Path
for z in sorted(Path('/content/aihub_raw').rglob('*.zip')):
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        print(f'{z.name}: {len(names)}개 항목, 예시 {names[:5]}')
        sample = next((n for n in names if n.lower().endswith('.json')), None)
        if sample:
            data = json.loads(zf.read(sample).decode('utf-8'))
            print('--- JSON 샘플:', sample)
            print(json.dumps(data, ensure_ascii=False, indent=1)[:2500])
print('xlsx:', [p.name for p in Path('/content/aihub_raw').rglob('*.xlsx')])

In [ ]:
# 4. YOLO 학습 형식으로 변환
# ⚠️ 이 셀은 74번 데이터셋용으로 아직 재작성 전이다 — 2b 출력(JSON 필드명·박스 좌표 형식·이미지 파일명 규칙)을 본 뒤
#    "원천 zip 한 덩어리 다운로드 → 압축 해제 → 목표 클래스만 640px 축소 저장 + JSON 박스 → YOLO 라벨 → 원본 삭제" 로 바꾼다.
#    아래는 이전 데이터셋(클래스별 폴더, 박스 없음)용 full-image bbox 변환이다. 74번에는 그대로 쓰지 않는다.
import random, shutil
from pathlib import Path

random.seed(SEED)
IMG_EXTS = {'.jpg', '.jpeg', '.png'}
src = Path(RAW_DIR)

# 이미지가 직접 들어 있는 말단 폴더를 클래스로 간주한다 (대분류/소분류 중첩 구조 모두 대응)
leaf_dirs = sorted(
    {f.parent for f in src.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXTS},
    key=lambda d: d.name,
)
if CLASS_LIMIT > 0:
    leaf_dirs = leaf_dirs[:CLASS_LIMIT]
class_names = [d.name for d in leaf_dirs]
assert len(class_names) == len(set(class_names)), '클래스 폴더명이 중복된다 — 소분류 폴더명이 유일한지 확인'
print(f'클래스 {len(class_names)}종: {class_names[:10]} ...')

root = Path('/content/food_yolo')
if root.exists():
    shutil.rmtree(root)
for split in ('train', 'val'):
    (root / 'images' / split).mkdir(parents=True)
    (root / 'labels' / split).mkdir(parents=True)

for cid, class_dir in enumerate(leaf_dirs):
    images = sorted(f for f in class_dir.iterdir() if f.suffix.lower() in IMG_EXTS)
    random.shuffle(images)
    if SAMPLES_PER_CLASS > 0:
        images = images[:SAMPLES_PER_CLASS]
    n_val = max(1, int(len(images) * VAL_RATIO))
    for i, img in enumerate(images):
        split = 'val' if i < n_val else 'train'
        # 파일명 충돌을 피하려고 클래스 id를 접두어로 붙인다
        dst_name = f'{cid:03d}_{img.name}'
        shutil.copy(img, root / 'images' / split / dst_name)
        # 사진 전체를 하나의 박스로: class cx cy w h (정규화 좌표)
        (root / 'labels' / split / f'{Path(dst_name).stem}.txt').write_text(f'{cid} 0.5 0.5 1.0 1.0\n')

data_yaml = root / 'data.yaml'
names_block = '\n'.join(f'  {i}: {n}' for i, n in enumerate(class_names))
data_yaml.write_text(f'path: {root}\ntrain: images/train\nval: images/val\nnames:\n{names_block}\n', encoding='utf-8')
print('train:', len(list((root / 'images' / 'train').iterdir())), '/ val:', len(list((root / 'images' / 'val').iterdir())))

In [ ]:
# 4. YOLO 학습 형식으로 변환 (full-image bbox)
import random, shutil
from pathlib import Path

random.seed(SEED)
IMG_EXTS = {'.jpg', '.jpeg', '.png'}
src = Path(RAW_DIR)

# 이미지가 직접 들어 있는 말단 폴더를 클래스로 간주한다 (대분류/소분류 중첩 구조 모두 대응)
leaf_dirs = sorted(
    {f.parent for f in src.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXTS},
    key=lambda d: d.name,
)
if CLASS_LIMIT > 0:
    leaf_dirs = leaf_dirs[:CLASS_LIMIT]
class_names = [d.name for d in leaf_dirs]
assert len(class_names) == len(set(class_names)), '클래스 폴더명이 중복된다 — 소분류 폴더명이 유일한지 확인'
print(f'클래스 {len(class_names)}종: {class_names[:10]} ...')

root = Path('/content/food_yolo')
if root.exists():
    shutil.rmtree(root)
for split in ('train', 'val'):
    (root / 'images' / split).mkdir(parents=True)
    (root / 'labels' / split).mkdir(parents=True)

for cid, class_dir in enumerate(leaf_dirs):
    images = sorted(f for f in class_dir.iterdir() if f.suffix.lower() in IMG_EXTS)
    random.shuffle(images)
    if SAMPLES_PER_CLASS > 0:
        images = images[:SAMPLES_PER_CLASS]
    n_val = max(1, int(len(images) * VAL_RATIO))
    for i, img in enumerate(images):
        split = 'val' if i < n_val else 'train'
        # 파일명 충돌을 피하려고 클래스 id를 접두어로 붙인다
        dst_name = f'{cid:03d}_{img.name}'
        shutil.copy(img, root / 'images' / split / dst_name)
        # 사진 전체를 하나의 박스로: class cx cy w h (정규화 좌표)
        (root / 'labels' / split / f'{Path(dst_name).stem}.txt').write_text(f'{cid} 0.5 0.5 1.0 1.0\n')

data_yaml = root / 'data.yaml'
names_block = '\n'.join(f'  {i}: {n}' for i, n in enumerate(class_names))
data_yaml.write_text(f'path: {root}\ntrain: images/train\nval: images/val\nnames:\n{names_block}\n', encoding='utf-8')
print('train:', len(list((root / 'images' / 'train').iterdir())), '/ val:', len(list((root / 'images' / 'val').iterdir())))

In [ ]:
# 5. YOLOv8n 파인튜닝 (T4 기준 30클래스 x 300장 x 60에폭 ≈ 1~2시간)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    seed=SEED,
    project='/content/runs',
    name='food',
)
BEST = '/content/runs/food/weights/best.pt'

In [ ]:
# 6. 검증 지표 확인 (mAP50이 0.8 이상이면 쓸 만하다)
metrics = YOLO(BEST).val(data=str(data_yaml))
print('mAP50:', round(metrics.box.map50, 3), '/ mAP50-95:', round(metrics.box.map, 3))

In [ ]:
# 8. 앱 호환성 자동 검증 — FoodDetector가 기대하는 조건과 대조
import numpy as np
import tensorflow as tf

interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]

assert inp['dtype'] == np.float32, f"입력이 {inp['dtype']} — 앱은 float32 입력만 받는다 (FoodDetector 로드 시 거부됨)"
assert out['dtype'] == np.float32, f"출력이 {out['dtype']} — float32여야 한다"
# 앱은 NHWC [1,H,W,3] 과 NCHW [1,3,H,W] 를 모두 처리한다. 어느 쪽인지 여기서 확인해 둔다.
shape = list(inp['shape'])
if shape == [1, IMG_SIZE, IMG_SIZE, 3]:
    layout = 'NHWC'
elif shape == [1, 3, IMG_SIZE, IMG_SIZE]:
    layout = 'NCHW'
else:
    raise AssertionError(f"입력 형태 {shape} — [1,{IMG_SIZE},{IMG_SIZE},3] 또는 [1,3,{IMG_SIZE},{IMG_SIZE}] 이어야 한다")
nc = len(class_names)
assert nc + 4 in list(out['shape']), f"출력 형태 {out['shape']}가 클래스 수 {nc}(+4)와 맞지 않는다"
print(f"검증 통과 — 입력 {shape} float32 ({layout}), 출력 {out['shape']} (클래스 {nc}종)")

In [ ]:
# 8. 앱 호환성 자동 검증 — FoodDetector가 기대하는 조건과 대조
import numpy as np
import tensorflow as tf

interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]

assert inp['dtype'] == np.float32, f"입력이 {inp['dtype']} — 앱은 float32 입력만 받는다 (FoodDetector 로드 시 거부됨)"
assert out['dtype'] == np.float32, f"출력이 {out['dtype']} — float32여야 한다"
assert list(inp['shape']) == [1, IMG_SIZE, IMG_SIZE, 3], f"입력 형태 {inp['shape']}"
nc = len(class_names)
assert nc + 4 in list(out['shape']), f"출력 형태 {out['shape']}가 클래스 수 {nc}(+4)와 맞지 않는다"
print(f"검증 통과 — 입력 {inp['shape']} float32, 출력 {out['shape']} (클래스 {nc}종)")

In [ ]:
# 9. 앱 배포용 파일 패키징 + 다운로드
import shutil
from google.colab import files

out_dir = Path('/content/app_assets')
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir()
shutil.copy(TFLITE_PATH, out_dir / 'yolov8n_food.tflite')
# 클래스 인덱스 순서 그대로 — 앱 food_labels.txt 형식 (전부 음식이므로 '#' 접두어 없음)
(out_dir / 'food_labels.txt').write_text('\n'.join(class_names) + '\n', encoding='utf-8')

shutil.make_archive('/content/trex_food_model', 'zip', out_dir)
files.download('/content/trex_food_model.zip')
print('완료 — zip 안의 두 파일을 TREX_UI/app/src/main/assets/models/ 에 덮어쓰고 빌드한다')

### 적용 방법
1. 받은 `trex_food_model.zip`을 풀어 `yolov8n_food.tflite`, `food_labels.txt` 두 파일을 `TREX_UI/app/src/main/assets/models/`에 **덮어쓴다** (기존 `food_labels.txt`는 COCO placeholder이므로 교체 대상이 맞다).
2. 앱을 빌드·설치하면 FoodDetector가 모델을 발견하고 시뮬레이션 폴백 없이 실추론으로 동작한다.
3. 인식은 되는데 이름이 앱 영양 DB(`foodDatabase`)에 없는 음식은 대략값으로 표시된다 — 식약처 영양 DB 연동 시 해소 예정.

### 문제 해결
- **GPU 세션 끊김**: Colab 무료는 세션 제한이 있다. `EPOCHS`를 줄이거나, `model.train(..., resume=True)`로 이어서 학습.
- **메모리 부족**: `BATCH`를 8로 줄인다.
- **정확도 낮음**: `SAMPLES_PER_CLASS`·`EPOCHS`를 늘리거나 `CLASS_LIMIT`을 줄여 클래스를 압축한다.
- **8번 셀 assert 실패(입출력이 float32가 아님)**: Ultralytics/onnx2tf 버전에 따라 INT8 변환 결과가 다를 수 있다. 7번 셀에서 `int8=True`를 `int8=False`로 바꿔 float32로 다시 변환한다 — 파일이 2~3배 커지지만 앱은 동일하게 동작한다. 이때 7번 셀의 산출물 검색 패턴도 `*int8*.tflite` 대신 `*.tflite`로 바꾼다.
- **다중 음식 인식 한계**: full-image bbox로 학습한 모델은 한 접시에 여러 음식이 담긴 사진에서 다중 인식 성능이 낮을 수 있다(사실상 사진 1장 = 음식 1종 학습). 여러 반찬을 한 번에 인식시키려면 반찬별로 나눠 찍거나, 추후 박스 라벨이 있는 데이터셋으로 재학습이 필요하다.